# Adverse events (`ae`) harmonization investigation

**Goal:** check whether the 56 columns shared across all 3 raw-domain trials (per
`column_comparison.ipynb`) actually mean the same thing and are coded the same way.

**Findings:**
1. **Severity/relatedness/seriousness fields are clean** — `AESEV`/`AESEVCD` (1-5 grade scale),
   `AEREL` (Y/N), `AESER` (Y/N), `SAELIFE` (Y/N) use identical coding across all 3 trials.
2. **MedDRA dictionary version differs**: PACCE uses **MedDRA v8**, the other two trials use
   **MedDRA v12**. MedDRA (Medical Dictionary for Regulatory Activities) is the standard coding
   dictionary trials use to classify adverse event terms into a hierarchy (System Organ Class ->
   High Level Group Term -> High Level Term -> Preferred Term -> Lowest Level Term). Different
   dictionary versions can reclassify or rename terms, so a version mismatch is a real harmonization
   risk for the coded term fields (`AESOC`, `AEPT`, `AEHLGT`, `AEHLT`, `AELLT`).
3. **Checked how much this actually matters**: at the System Organ Class level, PACCE's 25 SOC
   terms and the v12 trials' 25 SOC terms turned out to be a **pure case-formatting difference**
   (PACCE: ALL CAPS, v12 trials: Title Case) — 24 of 26 distinct SOC terms match once
   case-normalized. Only 2 genuine mismatches: `SOCIAL CIRCUMSTANCES` (PACCE only) and
   `PREGNANCY, PUERPERIUM AND PERINATAL CONDITIONS` (v12 trials only) — real MedDRA version
   artifacts, but rare/edge-case categories for a colorectal cancer trial population, not a
   practical modeling blocker.

**Recommendation:** harmonize `AESOC`/`AEPT`/`AEHLGT`/`AEHLT`/`AELLT` with `.str.upper().str.strip()`
normalization applied first. Keep `MEDDRA_V` as a passthrough column in the staging table so the
version difference is documented, not hidden — it's the kind of thing a reviewer with real MedDRA
familiarity would ask about. `AESEV`/`AESEVCD`, `AEREL`, `AESER`, `AESTDY`/`AEENDY`/`AEDUR` are
safe to harmonize with a straight rename.

In [1]:
import pyreadstat
from pathlib import Path
import pandas as pd

pd.set_option('display.max_rows', None)

DATA_DIR = Path('..') / '..' / 'Data' / 'raw'

TRIALS = {
    'PACCE (NCT00115225)': 'NCT00115225_PACCE_bev_panitumumab_raw',
    'FOLFIRI (NCT00339183)': 'NCT00339183_panitumumab_folfiri_raw',
    'FOLFOX/PRIME (NCT00364013)': 'NCT00364013_panitumumab_folfox_raw',
}

## 1. Severity / relatedness / seriousness fields — value comparison

**What these fields mean:**
- **`AESEV`/`AESEVCD`** — event severity grade: `1`=Mild, `2`=Moderate, `3`=Severe,
  `4`=Life threatening, `5`=Fatal. This is the CTCAE-style 1-5 grading oncology trials use, not
  a free-text severity description — same scale, same numbers, across all 3 trials.
- **`AEREL`** — "Related to Investigational Product?" (Y/N) — whether the event was assessed as
  causally related to the study drug, as opposed to disease progression or unrelated illness.
- **`AESER`** — "Serious?" (Y/N) — the regulatory definition of a Serious Adverse Event (SAE):
  results in death, is life-threatening, requires hospitalization, etc. Distinct from `AESEV`
  severity grade — a Mild event can still be flagged serious under specific regulatory criteria,
  though in practice they're highly correlated.
- **`SAELIFE`** — "Is Life-Threatening?" (Y/N) — one specific SAE sub-criterion, blank when not
  applicable/not an SAE.
- **`AESTDY`/`AEENDY`/`AEDUR`** — study day the event started/ended, and its duration in days.

In [2]:
key_cols = ['AESEV', 'AESEVCD', 'AEREL', 'AESER', 'SAELIFE']

for label, folder in TRIALS.items():
    path = DATA_DIR / folder / 'ae.sas7bdat'
    df, meta = pyreadstat.read_sas7bdat(str(path))
    label_lookup = dict(zip(meta.column_names, meta.column_labels))
    print(f'===== {label} — ae, n={len(df)} rows =====')
    for col in key_cols:
        lbl = label_lookup.get(col, '')
        print(f'-- {col} ({lbl}):')
        print(df[col].value_counts(dropna=False))
        print()
    print()

===== PACCE (NCT00115225) — ae, n=29080 rows =====
-- AESEV (Grade/Severity):
AESEV
Mild                    17894
Moderate                 7850
Severe                   2879
Life threatening          393
Fatal                      57
                   .        7
Name: count, dtype: int64

-- AESEVCD (Grade/Severity Code):
AESEVCD
1.0    17894
2.0     7850
3.0     2879
4.0      393
5.0       57
NaN        7
Name: count, dtype: int64

-- AEREL (Related to Investigational Product?):
AEREL
N    23990
Y     5090
Name: count, dtype: int64

-- AESER (Serious?):
AESER
N    27896
Y     1184
Name: count, dtype: int64

-- SAELIFE (Is Life-Threatening?):
SAELIFE
     28687
N      286
Y      107
Name: count, dtype: int64


===== FOLFIRI (NCT00339183) — ae, n=14842 rows =====
-- AESEV (Grade/Severity):
AESEV
Mild                8513
Moderate            4478
Severe              1510
Life threatening     234
Fatal                 62
                      45
Name: count, dtype: int64

-- AESEVCD (Grad

## 2. MedDRA dictionary version — does it actually break term comparability?

**What MedDRA is:** the Medical Dictionary for Regulatory Activities — the standard hierarchy
trials use to code adverse event terms consistently: System Organ Class (`AESOC`, broadest) ->
High Level Group Term (`AEHLGT`) -> High Level Term (`AEHLT`) -> Preferred Term (`AEPT`) ->
Lowest Level Term (`AELLT`, most granular) -> `AETERM` (the original reported free text, before
any coding). Sponsors periodically update to newer MedDRA dictionary versions, which can rename,
split, or reclassify terms — a real risk when pooling trials coded under different versions.

`MEDDRA_V` (dictionary version used) differs here: PACCE = v8, the other two trials = v12.

In [3]:
for label, folder in TRIALS.items():
    df, meta = pyreadstat.read_sas7bdat(str(DATA_DIR / folder / 'ae.sas7bdat'), usecols=['MEDDRA_V'])
    print(label, '-> MedDRA version(s):', df['MEDDRA_V'].unique())

PACCE (NCT00115225) -> MedDRA version(s): <StringArray>
['8.0']
Length: 1, dtype: str
FOLFIRI (NCT00339183) -> MedDRA version(s): <StringArray>
['12.0']
Length: 1, dtype: str
FOLFOX/PRIME (NCT00364013) -> MedDRA version(s): <StringArray>
['12.0']
Length: 1, dtype: str


**Checking whether the version difference actually matters** — compare the System Organ Class
(`AESOC`) term sets directly, then again after normalizing case, since a raw string-equality
comparison is misleading if the only difference is formatting, not classification.

In [4]:
socs = {}
for label, folder in TRIALS.items():
    df, meta = pyreadstat.read_sas7bdat(str(DATA_DIR / folder / 'ae.sas7bdat'), usecols=['AESOC'])
    socs[label] = set(df['AESOC'].dropna().unique())

shared_raw = socs['PACCE (NCT00115225)'] & socs['FOLFIRI (NCT00339183)'] & socs['FOLFOX/PRIME (NCT00364013)']
print(f'Shared SOC terms WITHOUT normalization: {len(shared_raw)}')
print()

socs_norm = {}
for label, folder in TRIALS.items():
    df, meta = pyreadstat.read_sas7bdat(str(DATA_DIR / folder / 'ae.sas7bdat'), usecols=['AESOC'])
    socs_norm[label] = set(df['AESOC'].dropna().str.upper().str.strip().unique()) - {''}

shared_norm = socs_norm['PACCE (NCT00115225)'] & socs_norm['FOLFIRI (NCT00339183)'] & socs_norm['FOLFOX/PRIME (NCT00364013)']
all_union = socs_norm['PACCE (NCT00115225)'] | socs_norm['FOLFIRI (NCT00339183)'] | socs_norm['FOLFOX/PRIME (NCT00364013)']
print(f'Shared SOC terms AFTER case-normalization: {len(shared_norm)} of {len(all_union)} distinct terms')
print('Genuine (non-formatting) mismatches:', all_union - shared_norm)

Shared SOC terms WITHOUT normalization: 0

Shared SOC terms AFTER case-normalization: 24 of 26 distinct terms
Genuine (non-formatting) mismatches: {'PREGNANCY, PUERPERIUM AND PERINATAL CONDITIONS', 'SOCIAL CIRCUMSTANCES'}


**Conclusion:** the version difference is mostly a formatting artifact, not a real
classification break, at the SOC level. The 2 genuine leftover mismatches
(`SOCIAL CIRCUMSTANCES`, `PREGNANCY, PUERPERIUM AND PERINATAL CONDITIONS`) are rare/edge-case
categories for this patient population — worth documenting in the dbt model's comments, not
worth blocking on. Finer-grained levels (`AEPT`, `AELLT`) weren't re-checked here since they're
much higher-cardinality (800-1100 distinct Preferred Terms per trial) — if a later modeling stage
needs AE-term-level features (rather than just SOC-level), that comparison should be redone at
that point before trusting term-level joins across the v8/v12 boundary.